In [1]:
from icalens import ICALens

lens = ICALens.from_pretrained("sida/icalens-gpt2-small-pile10k")
result = lens.analyze("She deposited the check at the bank.", layer=6)

result

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

AnalysisResult(tokens=('She', 'Ġdeposited', 'Ġthe', 'Ġcheck', 'Ġat', 'Ġthe', 'Ġbank', '.'), token_texts=('She', ' deposited', ' the', ' check', ' at', ' the', ' bank', '.'), token_labels=('She', ' deposited', ' the', ' check', ' at', ' the', ' bank', '.'), token_tooltips=('She', 'Ġdeposited', 'Ġthe', 'Ġcheck', 'Ġat', 'Ġthe', 'Ġbank', '.'), token_groups=(), token_ids=tensor([ 3347, 27163,   262,  2198,   379,   262,  3331,    13]), positions=tensor([0, 1, 2, 3, 4, 5, 6, 7]), activations=tensor([[-5.5664e-02, -9.0625e-01,  7.0312e-01,  ..., -1.2031e+00,
          4.3945e-03, -6.1328e-01],
        [ 2.3438e+00, -1.2695e-02, -8.9062e-01,  ...,  1.7578e-01,
          2.4844e+00, -6.8359e-01],
        [ 2.6875e+00,  1.4297e+00, -2.9297e-03,  ..., -1.6250e+00,
          1.8828e+00, -1.0156e+00],
        ...,
        [ 2.1289e-01,  8.3984e-02, -1.2578e+00,  ...,  1.2266e+00,
         -1.3750e+00,  3.2812e-01],
        [ 1.8750e-01,  4.0234e-01, -3.5000e+00,  ...,  8.3984e-01,
         -6.0156e-01, -1.2109e+00],
        [ 2.2656e+00, -3.9844e+00, -1.5156e+00,  ...,  4.4336e-01,
          1.6992e-01, -1.2031e+00]], device='cuda:0'), scores=tensor([[-0.0104, -0.0144,  0.0306,  ...,  0.0324, -0.0114,  0.0040],
        [-0.1711,  0.5289,  1.1646,  ..., -0.0703,  0.8614,  0.1774],
        [-0.1707,  0.2794,  0.3521,  ...,  0.9673,  1.8925, -0.0027],
        ...,
        [ 0.3699, -0.0694,  0.6804,  ...,  0.3498,  1.0682, -0.5139],
        [-0.3857, -0.0021,  1.4710,  ...,  0.1975,  0.4185, -0.8218],
        [ 0.4959, -0.6924,  0.6152,  ..., -0.1189,  0.0153, -0.4901]],
       device='cuda:0'), energy=tensor([[2.0840e-07, 4.0572e-07, 1.8212e-06,  ..., 2.0384e-06, 2.5370e-07,
         3.1274e-08],
        [4.9048e-05, 4.6883e-04, 2.2733e-03,  ..., 8.2830e-06, 1.2437e-03,
         5.2761e-05],
        [6.6714e-05, 1.7872e-04, 2.8384e-04,  ..., 2.1418e-03, 8.1982e-03,
         1.6311e-08],
        ...,
        [2.4382e-04, 8.5684e-06, 8.2472e-04,  ..., 2.1801e-04, 2.0328e-03,
         4.7058e-04],
        [2.0415e-04, 6.1995e-09, 2.9701e-03,  ..., 5.3528e-05, 2.4045e-04,
         9.2700e-04],
        [5.8850e-04, 1.1473e-03, 9.0564e-04,  ..., 3.3839e-05, 5.6302e-07,
         5.7489e-04]], device='cuda:0'), model='openai-community/gpt2@607a30d783dfa663caf39e06633721c8d4cfcd7e', layer=6, input_text='She deposited the check at the bank.', token_scope='all text tokens', messages=())

In [3]:
top_scores = lens.keep_topk(result.scores, k=10)
top_reconstruction = lens.inverse_transform(top_scores, layer=6)
reconstructed = lens.restore_norm(
    top_reconstruction,
    reference=result.activations,
)

In [5]:
import torch.nn.functional as F

similarity = F.cosine_similarity(
    F.normalize(result.activations, dim=-1),
    top_reconstruction,
    dim=-1,
)
similarity

tensor([0.9997, 0.8574, 0.9278, 0.8388, 0.8962, 0.9238, 0.8575, 0.9378],
       device='cuda:0')

In [9]:
normalized = F.normalize(result.activations, dim=-1)
baseline = normalized.mean(dim=0, keepdim=True)

token_squared_error = (normalized - top_reconstruction).square().sum(dim=-1)
token_baseline_error = (normalized - baseline).square().sum(dim=-1)
token_normalized_mse = token_squared_error / token_baseline_error.clamp_min(1e-12)

print(token_normalized_mse)  # one value per token

tensor([0.0012, 1.1072, 0.6597, 1.0565, 0.7393, 0.6087, 0.8133, 0.4797],
       device='cuda:0')
